# Tomek Links + 모델 2개 성능 비교

이 notebook은 `docs/기법별_성능평가.md`의 평가 계획에 맞춰 `Undersampling Tomek Links + 모델 2개` 실험을 준비하는 템플릿입니다.


## 목적

클래스 경계에서 서로 가까운 Tomek Links를 제거해 모델 1의 결정경계를 더 명확하게 만드는지 확인한다.

TomekLinks는 클래스 비율을 1:1로 강제하지 않고 경계 샘플을 정리하는 undersampling 기법이다.


## 필요한 입력 파일

- `data/processed/original_train_train_preprocessed.csv`
- `data/processed/original_train_val_preprocessed.csv`
- `data/processed/original_train_test_preprocessed.csv`
- `artifacts/preprocessors/original_train_preprocess_config.json`

## 실행 절차

1. 입력 파일과 feature 목록을 확인한다.
2. 모델 1 학습용 train split에만 imbalanced-learn sampling을 적용한다.
3. sampling된 train CSV를 저장한다.
4. 모델 1은 sampling된 train CSV로 학습한다.
5. 모델 2는 sampling을 적용하지 않고, 기존 스크립트가 `위험도 > 0` 데이터만 필터링하여 학습한다.
6. 결과 지표를 계산하고 `docs/기법별_성능평가.md` 평가표에 옮겨 적는다.

## 모델 2 주의사항

모델 2 회귀모델은 항상 `위험도 > 0` 데이터만 사용한다. Sampling 기법은 모델 1의 `사고발생` 분류 문제에만 적용한다.


In [1]:
from __future__ import annotations

import json
import math
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, mean_absolute_error, mean_squared_error

from imblearn.under_sampling import TomekLinks

PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'src').exists()
)

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
TRAIN_PATH = DATA_DIR / 'original_train_train_preprocessed.csv'
VAL_PATH = DATA_DIR / 'original_train_val_preprocessed.csv'
TEST_PATH = DATA_DIR / 'original_train_test_preprocessed.csv'
PREPROCESS_CONFIG_PATH = PROJECT_ROOT / 'artifacts' / 'preprocessors' / 'original_train_preprocess_config.json'

EXPERIMENT_NAME = 'Undersampling Tomek Links + 모델 2개'
EXPERIMENT_SLUG = 'tomek_links_two_stage'
RANDOM_STATE = 42
ROW_LIMIT = None  # None이면 전체 train 데이터를 사용한다.
RUN_SAMPLING = True  # 실제 sampling CSV를 만들려면 True로 변경한다.
RUN_TRAINING = True  # 실제 모델 학습까지 실행하려면 True로 변경한다.

OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'experiments' / EXPERIMENT_SLUG
SAMPLED_INPUT_DIR = PROJECT_ROOT / 'artifacts' / 'experiments' / 'sampling_inputs'
SAMPLED_TRAIN_PATH = SAMPLED_INPUT_DIR / f'{EXPERIMENT_SLUG}_model1_train_preprocessed.csv'

MODEL1_MODEL_PATH = PROJECT_ROOT / 'artifacts' / 'models' / f'mlp_accident_classifier_{EXPERIMENT_SLUG}.keras'
MODEL1_METRICS_PATH = PROJECT_ROOT / 'artifacts' / 'reports' / f'mlp_accident_classifier_{EXPERIMENT_SLUG}_metrics.json'
MODEL1_HISTORY_PATH = PROJECT_ROOT / 'artifacts' / 'reports' / f'mlp_accident_classifier_{EXPERIMENT_SLUG}_history.csv'
MODEL1_PREDICTIONS_PATH = PROJECT_ROOT / 'artifacts' / 'predictions' / f'mlp_accident_classifier_{EXPERIMENT_SLUG}_test_predictions.csv'

MODEL2_MODEL_PATH = PROJECT_ROOT / 'artifacts' / 'models' / f'mlp_positive_risk_regressor_{EXPERIMENT_SLUG}.keras'
MODEL2_METRICS_PATH = PROJECT_ROOT / 'artifacts' / 'reports' / f'mlp_positive_risk_regressor_{EXPERIMENT_SLUG}_metrics.json'
MODEL2_HISTORY_PATH = PROJECT_ROOT / 'artifacts' / 'reports' / f'mlp_positive_risk_regressor_{EXPERIMENT_SLUG}_history.csv'
MODEL2_PREDICTIONS_PATH = PROJECT_ROOT / 'artifacts' / 'predictions' / f'mlp_positive_risk_regressor_{EXPERIMENT_SLUG}_test_predictions.csv'

EXPERIMENT_RESULT_PATH = OUTPUT_DIR / f'{EXPERIMENT_SLUG}_summary.csv'
DOCS_TABLE_PATH = PROJECT_ROOT / 'docs' / '기법별_성능평가.md'


In [2]:
required_paths = {
    'train': TRAIN_PATH,
    'val': VAL_PATH,
    'test': TEST_PATH,
    'preprocess_config': PREPROCESS_CONFIG_PATH,
    'docs_table': DOCS_TABLE_PATH,
}

path_status = pd.DataFrame(
    [{'name': name, 'path': str(path.relative_to(PROJECT_ROOT)), 'exists': path.exists()} for name, path in required_paths.items()]
)
display(path_status)

missing = path_status.loc[~path_status['exists'], 'path'].tolist()
if missing:
    raise FileNotFoundError(f'필요한 파일이 없습니다: {missing}')

with PREPROCESS_CONFIG_PATH.open('r', encoding='utf-8') as file:
    preprocess_config = json.load(file)

feature_columns = preprocess_config['feature_columns']
print(f'feature 수: {len(feature_columns):,}')


,name,path,exists
0,train,data/processed/original_train_train_preprocess...,True
1,val,data/processed/original_train_val_preprocessed...,True
2,test,data/processed/original_train_test_preprocesse...,True
3,preprocess_config,artifacts/preprocessors/original_train_preproc...,True
4,docs_table,docs/기법별_성능평가.md,True


feature 수: 102


In [3]:
def accident_target(frame: pd.DataFrame) -> pd.Series:
    """위험도 값을 모델 1의 사고발생 이진 라벨로 변환한다."""
    return (frame['위험도'] > 0).astype(int)


train_frame = pd.read_csv(TRAIN_PATH, nrows=ROW_LIMIT)
X_train = train_frame[feature_columns]
y_train = accident_target(train_frame)

print('sampling 실행 전 클래스 분포')
display(y_train.value_counts().rename(index={0: '사고발생=0', 1: '사고발생=1'}).to_frame('count'))

sampler = TomekLinks(n_jobs=-1)
print(sampler)

if RUN_SAMPLING:
    X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)
    sampled_train = pd.DataFrame(X_resampled, columns=feature_columns)

    # 모델 1 학습 스크립트는 위험도 > 0 여부만 사용하므로, sampling 후 라벨 복원을 위해 위험도 컬럼을 만든다.
    sampled_train['위험도'] = np.where(np.asarray(y_resampled) == 1, 1.0, 0.0)

    SAMPLED_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    sampled_train.to_csv(SAMPLED_TRAIN_PATH, index=False)

    print('sampling 실행 후 클래스 분포')
    display(pd.Series(y_resampled).value_counts().rename(index={0: '사고발생=0', 1: '사고발생=1'}).to_frame('count'))
    print(f'sampled train 저장: {SAMPLED_TRAIN_PATH.relative_to(PROJECT_ROOT)}')
else:
    print('RUN_SAMPLING=False 상태입니다. 실제 sampling CSV 생성은 실행하지 않았습니다.')
    print('실행하려면 위 설정 셀에서 RUN_SAMPLING=True로 변경하십시오.')


sampling 실행 전 클래스 분포


,count
위험도,
사고발생=0,252551
사고발생=1,62913


TomekLinks(n_jobs=-1)


/tmp/ipykernel_421650/493377969.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sampled_train['위험도'] = np.where(np.asarray(y_resampled) == 1, 1.0, 0.0)


sampling 실행 후 클래스 분포


,count
위험도,
사고발생=0,247989
사고발생=1,62913


sampled train 저장: artifacts/experiments/sampling_inputs/tomek_links_two_stage_model1_train_preprocessed.csv


In [4]:
model1_command = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'train' / 'train_accident_classifier.py'),
    '--train', str(SAMPLED_TRAIN_PATH),
    '--val', str(VAL_PATH),
    '--test', str(TEST_PATH),
    '--model-path', str(MODEL1_MODEL_PATH),
    '--metrics-path', str(MODEL1_METRICS_PATH),
    '--history-path', str(MODEL1_HISTORY_PATH),
    '--predictions-path', str(MODEL1_PREDICTIONS_PATH),
    '--epochs', '100',
    '--batch-size', '1024',
    '--device', 'auto',
    '--verbose', '2',
    '--top-k', '1000',
]

print('모델 1 학습 명령:')
print(' '.join(model1_command))

if RUN_TRAINING:
    if not SAMPLED_TRAIN_PATH.exists():
        raise FileNotFoundError('모델 1 학습 전 RUN_SAMPLING=True로 sampling train CSV를 먼저 생성해야 합니다.')
    subprocess.run(model1_command, check=True)
else:
    print('RUN_TRAINING=False 상태입니다. 실제 모델 1 학습은 실행하지 않았습니다.')


모델 1 학습 명령:
/home/huichan/SPARV/SilverWalk/.venv/bin/python /home/huichan/SPARV/SilverWalk/scripts/train/train_accident_classifier.py --train /home/huichan/SPARV/SilverWalk/artifacts/experiments/sampling_inputs/tomek_links_two_stage_model1_train_preprocessed.csv --val /home/huichan/SPARV/SilverWalk/data/processed/original_train_val_preprocessed.csv --test /home/huichan/SPARV/SilverWalk/data/processed/original_train_test_preprocessed.csv --model-path /home/huichan/SPARV/SilverWalk/artifacts/models/mlp_accident_classifier_tomek_links_two_stage.keras --metrics-path /home/huichan/SPARV/SilverWalk/artifacts/reports/mlp_accident_classifier_tomek_links_two_stage_metrics.json --history-path /home/huichan/SPARV/SilverWalk/artifacts/reports/mlp_accident_classifier_tomek_links_two_stage_history.csv --predictions-path /home/huichan/SPARV/SilverWalk/artifacts/predictions/mlp_accident_classifier_tomek_links_two_stage_test_predictions.csv --epochs 100 --batch-size 1024 --device auto --verbose 2 --top

I0000 00:00:1780779705.986841  422005 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780779706.053144  422005 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780779706.893930  422005 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1780779707.709137  422005 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please mak

Epoch 1/100
304/304 - 2s - 8ms/step - accuracy: 0.8050 - loss: 0.4399 - pr_auc: 0.4356 - precision: 0.5428 - recall: 0.2313 - roc_auc: 0.7493 - val_accuracy: 0.8199 - val_loss: 0.4109 - val_pr_auc: 0.5024 - val_precision: 0.6291 - val_recall: 0.2358 - val_roc_auc: 0.7857 - learning_rate: 0.0010
Epoch 2/100
304/304 - 1s - 4ms/step - accuracy: 0.8171 - loss: 0.4150 - pr_auc: 0.5016 - precision: 0.6144 - recall: 0.2589 - roc_auc: 0.7831 - val_accuracy: 0.8243 - val_loss: 0.4004 - val_pr_auc: 0.5265 - val_precision: 0.6621 - val_recall: 0.2432 - val_roc_auc: 0.7997 - learning_rate: 0.0010
Epoch 3/100
304/304 - 1s - 4ms/step - accuracy: 0.8213 - loss: 0.4060 - pr_auc: 0.5223 - precision: 0.6310 - recall: 0.2811 - roc_auc: 0.7954 - val_accuracy: 0.8276 - val_loss: 0.3920 - val_pr_auc: 0.5457 - val_precision: 0.6744 - val_recall: 0.2616 - val_roc_auc: 0.8112 - learning_rate: 0.0010
Epoch 4/100
304/304 - 1s - 4ms/step - accuracy: 0.8242 - loss: 0.3986 - pr_auc: 0.5386 - precision: 0.6411 - rec

In [5]:
model2_command = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'train' / 'train_positive_risk_regressor.py'),
    '--train', str(TRAIN_PATH),
    '--val', str(VAL_PATH),
    '--test', str(TEST_PATH),
    '--model-path', str(MODEL2_MODEL_PATH),
    '--metrics-path', str(MODEL2_METRICS_PATH),
    '--history-path', str(MODEL2_HISTORY_PATH),
    '--predictions-path', str(MODEL2_PREDICTIONS_PATH),
    '--epochs', '100',
    '--batch-size', '1024',
    '--device', 'auto',
    '--verbose', '2',
]

print('모델 2 학습 명령:')
print(' '.join(model2_command))
print('주의: 모델 2 학습 스크립트는 내부에서 위험도 > 0 데이터만 필터링합니다.')

if RUN_TRAINING:
    subprocess.run(model2_command, check=True)
else:
    print('RUN_TRAINING=False 상태입니다. 실제 모델 2 학습은 실행하지 않았습니다.')


모델 2 학습 명령:
/home/huichan/SPARV/SilverWalk/.venv/bin/python /home/huichan/SPARV/SilverWalk/scripts/train/train_positive_risk_regressor.py --train /home/huichan/SPARV/SilverWalk/data/processed/original_train_train_preprocessed.csv --val /home/huichan/SPARV/SilverWalk/data/processed/original_train_val_preprocessed.csv --test /home/huichan/SPARV/SilverWalk/data/processed/original_train_test_preprocessed.csv --model-path /home/huichan/SPARV/SilverWalk/artifacts/models/mlp_positive_risk_regressor_tomek_links_two_stage.keras --metrics-path /home/huichan/SPARV/SilverWalk/artifacts/reports/mlp_positive_risk_regressor_tomek_links_two_stage_metrics.json --history-path /home/huichan/SPARV/SilverWalk/artifacts/reports/mlp_positive_risk_regressor_tomek_links_two_stage_history.csv --predictions-path /home/huichan/SPARV/SilverWalk/artifacts/predictions/mlp_positive_risk_regressor_tomek_links_two_stage_test_predictions.csv --epochs 100 --batch-size 1024 --device auto --verbose 2
주의: 모델 2 학습 스크립트는 내부에서

I0000 00:00:1780779845.199153  426502 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780779845.234304  426502 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780779846.010357  426502 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1780779846.784689  426502 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please mak

Epoch 1/100
62/62 - 1s - 17ms/step - loss: 0.3960 - mae: 0.7495 - rmse: 1.0024 - val_loss: 0.3521 - val_mae: 0.6807 - val_rmse: 0.9725 - learning_rate: 0.0010
Epoch 2/100
62/62 - 0s - 5ms/step - loss: 0.3073 - mae: 0.6399 - rmse: 0.8638 - val_loss: 0.2889 - val_mae: 0.6090 - val_rmse: 0.8574 - learning_rate: 0.0010
Epoch 3/100
62/62 - 0s - 5ms/step - loss: 0.2888 - mae: 0.6157 - rmse: 0.8339 - val_loss: 0.2681 - val_mae: 0.5835 - val_rmse: 0.8134 - learning_rate: 0.0010
Epoch 4/100
62/62 - 0s - 5ms/step - loss: 0.2798 - mae: 0.6044 - rmse: 0.8196 - val_loss: 0.2569 - val_mae: 0.5708 - val_rmse: 0.7883 - learning_rate: 0.0010
Epoch 5/100
62/62 - 0s - 5ms/step - loss: 0.2717 - mae: 0.5938 - rmse: 0.8050 - val_loss: 0.2491 - val_mae: 0.5609 - val_rmse: 0.7724 - learning_rate: 0.0010
Epoch 6/100
62/62 - 0s - 5ms/step - loss: 0.2651 - mae: 0.5852 - rmse: 0.7940 - val_loss: 0.2446 - val_mae: 0.5554 - val_rmse: 0.7635 - learning_rate: 0.0010
Epoch 7/100
62/62 - 0s - 5ms/step - loss: 0.2601 - 

## 결과 지표 표 위치

실험을 실행한 뒤 아래 지표만 `docs/기법별_성능평가.md`의 평가표에 작성한다.

| 모델 | PR-AUC | Recall@Top10% | Precision@Top10% | MAE | RMSE |
| --- | ---: | ---: | ---: | ---: | ---: |
| 실험명 | 실행 후 입력 | 실행 후 입력 | 실행 후 입력 | 실행 후 입력 | 실행 후 입력 |


In [6]:
def top_percent_binary_metrics(predictions: pd.DataFrame, score_column: str, target_column: str, percent: float = 0.10) -> dict[str, float]:
    """예측 점수 상위 percent 구간의 precision/recall을 계산한다."""
    if predictions.empty:
        return {'precision_top10pct': math.nan, 'recall_top10pct': math.nan}

    k = max(1, math.ceil(len(predictions) * percent))
    ranked = predictions.sort_values(score_column, ascending=False).head(k)
    actual_positive = predictions[target_column].astype(int)
    top_positive = ranked[target_column].astype(int)

    precision = float(top_positive.sum() / k)
    recall = float(top_positive.sum() / actual_positive.sum()) if actual_positive.sum() else math.nan
    return {'precision_top10pct': precision, 'recall_top10pct': recall}


def rmse(y_true: pd.Series, y_pred: pd.Series) -> float:
    """scikit-learn 버전 차이를 피하기 위해 RMSE를 직접 계산한다."""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


if MODEL1_PREDICTIONS_PATH.exists() and MODEL2_PREDICTIONS_PATH.exists():
    model1_predictions = pd.read_csv(MODEL1_PREDICTIONS_PATH)
    model2_predictions = pd.read_csv(MODEL2_PREDICTIONS_PATH)

    model1_summary = top_percent_binary_metrics(
        model1_predictions,
        score_column='사고발생확률',
        target_column='사고발생',
        percent=0.10,
    )
    model1_summary['pr_auc'] = float(
        average_precision_score(model1_predictions['사고발생'], model1_predictions['사고발생확률'])
    )

    model2_summary = {
        'mae': float(mean_absolute_error(model2_predictions['위험도'], model2_predictions['pred_위험도'])),
        'rmse': rmse(model2_predictions['위험도'], model2_predictions['pred_위험도']),
    }

    summary = pd.DataFrame([
        {
            '모델': EXPERIMENT_NAME,
            'PR-AUC': model1_summary['pr_auc'],
            'Recall@Top10%': model1_summary['recall_top10pct'],
            'Precision@Top10%': model1_summary['precision_top10pct'],
            'MAE': model2_summary['mae'],
            'RMSE': model2_summary['rmse'],
        }
    ])

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary.to_csv(EXPERIMENT_RESULT_PATH, index=False)
    display(summary)
    print(f'실험 요약 저장: {EXPERIMENT_RESULT_PATH.relative_to(PROJECT_ROOT)}')
    print(f'문서 평가표 위치: {DOCS_TABLE_PATH.relative_to(PROJECT_ROOT)}')
else:
    print('아직 모델 예측 파일이 없습니다. RUN_SAMPLING=True, RUN_TRAINING=True로 학습을 마친 뒤 다시 실행하십시오.')
    print(f'필요 파일: {MODEL1_PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}')
    print(f'필요 파일: {MODEL2_PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}')


,모델,PR-AUC,Recall@Top10%,Precision@Top10%,MAE,RMSE
0,Undersampling Tomek Links + 모델 2개,0.738218,0.41077,0.819231,2.039103,6.307999


실험 요약 저장: artifacts/experiments/tomek_links_two_stage/tomek_links_two_stage_summary.csv
문서 평가표 위치: docs/기법별_성능평가.md
